# BERT-SSIN ABSA  SemEval-2014 

**Aspect-Based Sentiment Analysis** sử dụng BERT + Syncretic Info Network (SSIN) + Semantic Guided Self-Attention (SGSA).

****
-  Dùng file test XML gốc (`Restaurants_Test_Gold.xml`) đúng benchmark
-  Align dependency adjacency matrix với BERT subword tokens
-  Focal Loss thay CrossEntropyLoss để xử lý class imbalance
-  Early stopping dựa trên Macro-F1
-  Best model lưu theo F1 thay vì Accuracy

---

## 1. Cài đặt thư viện

In [ ]:
#  Cài đặt 
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'scikit-learn', 'spacy'])
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'])

import transformers
print(f'transformers {transformers.__version__}')

## 2. Download dữ liệu SemEval-2014

Download trực tiếp từ GitHub repo [`nguyenmaiductrong/absa-sota-survey`](https://github.com/nguyenmaiductrong/absa-sota-survey).  
Bao gồm cả file test gold gốc cho Restaurant và Laptop.

In [ ]:
#  Download SemEval-2014 data từ GitHub 
import os, urllib.request

os.makedirs('/kaggle/working/semeval14', exist_ok=True)

FILES = {
    'Restaurants_Train_v2.xml':  'https://raw.githubusercontent.com/nguyenmaiductrong/absa-sota-survey/main/data/raw/semeval14/Restaurants_Train_v2.xml',
    'Restaurants_Test_Gold.xml': 'https://raw.githubusercontent.com/nguyenmaiductrong/absa-sota-survey/main/data/raw/semeval14/Restaurants_Test_Gold.xml',
    'Laptop_Train_v2.xml':       'https://raw.githubusercontent.com/nguyenmaiductrong/absa-sota-survey/main/data/raw/semeval14/Laptop_Train_v2.xml',
    'Laptops_Test_Gold.xml':     'https://raw.githubusercontent.com/nguyenmaiductrong/absa-sota-survey/main/data/raw/semeval14/Laptops_Test_Gold.xml',
}

for fname, url in FILES.items():
    dest = f'/kaggle/working/semeval14/{fname}'
    if not os.path.exists(dest):
        print(f'  Downloading {fname}...')
        urllib.request.urlretrieve(url, dest)
        print(f' {fname} saved')
    else:
        print(f' {fname} already exists')

print('\n Tất cả file đã sẵn sàng!')

## 3. Parse XML & chuẩn bị dữ liệu

- Đọc file XML SemEval-2014
- Lọc bỏ nhãn `conflict`
- Fallback về 80/20 split nếu không tìm thấy file test gold

In [ ]:
#  Parse XML  dùng file test gốc SemEval-2014 
import os, xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

# Dataset charitarth có cả file test gold
DATA_ROOT = '/kaggle/working/semeval14'
os.makedirs('data', exist_ok=True)

POL_MAP = {'positive': 2, 'neutral': 1, 'negative': 0}

def xml_to_samples(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    samples = []
    for sent in root.iter('sentence'):
        text_el = sent.find('text')
        if text_el is None:
            continue
        text = (text_el.text or '').strip().lower()
        asp_terms = sent.find('aspectTerms')
        if asp_terms is None:
            continue
        for at in asp_terms.findall('aspectTerm'):
            aspect = (at.get('term') or '').strip().lower()
            pol    = (at.get('polarity') or '').strip().lower()
            if not aspect or pol == 'conflict':
                continue
            pol_int = POL_MAP.get(pol)
            if pol_int is None:
                continue
            samples.append({'text': text, 'aspect': aspect, 'polarity': pol_int})
    return samples

def save_raw(samples, path):
    with open(path, 'w', encoding='utf-8') as f:
        for s in samples:
            f.write(f"{s['text']}\n{s['aspect']}\n{s['polarity']}\n")

# Restaurant: dùng train + test gold gốc
RESTAURANT_TRAIN = f'{DATA_ROOT}/Restaurants_Train_v2.xml'
RESTAURANT_TEST  = f'{DATA_ROOT}/Restaurants_Test_Gold.xml'   #  file test gốc
LAPTOP_TRAIN     = f'{DATA_ROOT}/Laptop_Train_v2.xml'
LAPTOP_TEST      = f'{DATA_ROOT}/Laptops_Test_Gold.xml'       #  file test gốc

for domain, train_file, test_file in [
    ('restaurant', RESTAURANT_TRAIN, RESTAURANT_TEST),
    ('laptop',     LAPTOP_TRAIN,     LAPTOP_TEST),
]:
    train_samples = xml_to_samples(train_file)
    # Nếu file test gold không tồn tại  fallback về 80/20 split
    if os.path.exists(test_file):
        test_samples = xml_to_samples(test_file)
        print(f' {domain}: {len(train_samples)} train | {len(test_samples)} test   gold test file')
    else:
        train_samples, test_samples = train_test_split(
            train_samples, test_size=0.2, random_state=42,
            stratify=[s['polarity'] for s in train_samples])
        print(f'  {domain}: gold test not found  80/20 split | {len(train_samples)} train | {len(test_samples)} test')
    save_raw(train_samples, f'data/{domain}_train.raw')
    save_raw(test_samples,  f'data/{domain}_test.raw')

print('\n Dữ liệu sẵn sàng!')

## 4. Dataset, Tokenizer & Dependency Adjacency Matrix

**Điểm quan trọng:** Hàm `build_dep_adj_bert` align adjacency matrix với BERT subword tokens.  
Mỗi word được map sang các subword tokens tương ứng, edges được copy sang tất cả subword token pairs.

In [ ]:
#  Dataset & tokenizer 
import numpy as np
import torch
import spacy
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from collections import Counter

nlp = spacy.load('en_core_web_sm')
print(' spaCy loaded')

BERT_MODEL = 'bert-base-uncased'
tokenizer  = BertTokenizer.from_pretrained(BERT_MODEL)
print(f' BERT tokenizer: {BERT_MODEL}')

def read_raw(path):
    samples = []
    with open(path, encoding='utf-8') as f:
        lines = [l.strip() for l in f]
    for i in range(0, len(lines) - 2, 3):
        text     = lines[i].lower()
        aspect   = lines[i+1].lower()
        polarity = int(lines[i+2])
        samples.append({'text': text, 'aspect': aspect, 'polarity': polarity})
    return samples

#  FIX: Align dependency adj với BERT subword tokens 
def build_dep_adj_bert(text, tokenizer, max_len):
    """
    Build adjacency matrix aligned với BERT token positions.
    Mỗi word được map sang các subword tokens tương ứng,
    edges giữa words được copy sang tất cả subword token pairs.
    """
    adj = np.eye(max_len, dtype=np.float32)
    try:
        doc = nlp(text)
        words = [tok.text for tok in doc]

        # Map word index  list of BERT subword token indices
        # tokenizer encode từng word riêng để biết nó sinh ra bao nhiêu subword
        word_to_tokens = {}
        cur_pos = 1  # bắt đầu từ 1 vì token 0 là [CLS]
        for wi, word in enumerate(words):
            sub = tokenizer.tokenize(word)
            n   = len(sub)
            if cur_pos + n - 1 < max_len:
                word_to_tokens[wi] = list(range(cur_pos, cur_pos + n))
            cur_pos += n
            if cur_pos >= max_len - 1:  # -1 cho [SEP]
                break

        # Copy edges từ word-level sang token-level
        for tok in doc:
            wi, wj = tok.i, tok.head.i
            if wi in word_to_tokens and wj in word_to_tokens:
                for ti in word_to_tokens[wi]:
                    for tj in word_to_tokens[wj]:
                        if ti < max_len and tj < max_len:
                            adj[ti][tj] = adj[tj][ti] = 1.0
    except Exception:
        # Fallback: window-based adjacency
        words = text.split()
        cur = 1
        positions = []
        for w in words:
            sub = tokenizer.tokenize(w)
            n   = len(sub)
            positions.append(list(range(cur, min(cur + n, max_len))))
            cur += n
            if cur >= max_len - 1:
                break
        for i, pi in enumerate(positions):
            for j in range(max(0, i - 3), min(len(positions), i + 4)):
                pj = positions[j]
                for ti in pi:
                    for tj in pj:
                        if ti < max_len and tj < max_len:
                            adj[ti][tj] = adj[tj][ti] = 1.0
    return adj

class BertABSADataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=128):
        self.data = []
        for s in samples:
            enc = tokenizer(
                s['text'], s['aspect'],
                max_length=max_len, padding='max_length',
                truncation=True, return_tensors='pt'
            )
            input_ids      = enc['input_ids'].squeeze(0)
            attention_mask = enc['attention_mask'].squeeze(0)
            token_type_ids = enc['token_type_ids'].squeeze(0)

            # Aspect mask: tokens giữa [SEP] đầu và [SEP] cuối
            sep_pos   = (input_ids == tokenizer.sep_token_id).nonzero(as_tuple=True)[0]
            asp_start = sep_pos[0].item() + 1 if len(sep_pos) > 0 else 0
            asp_end   = sep_pos[1].item()     if len(sep_pos) > 1 else max_len
            asp_mask  = torch.zeros(max_len)
            asp_mask[asp_start:asp_end] = 1.0

            # FIX: adj aligned với BERT tokens
            adj = torch.tensor(
                build_dep_adj_bert(s['text'], tokenizer, max_len),
                dtype=torch.float
            )
            self.data.append({
                'input_ids':      input_ids,
                'attention_mask': attention_mask,
                'token_type_ids': token_type_ids,
                'asp_mask':       asp_mask,
                'adj':            adj,
                'polarity':       torch.tensor(s['polarity'], dtype=torch.long),
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]


DATASET    = 'restaurant'
MAX_LEN    = 128
BATCH_SIZE = 16

train_samples = read_raw(f'data/{DATASET}_train.raw')
test_samples  = read_raw(f'data/{DATASET}_test.raw')
print(f' Train: {len(train_samples)} | Test: {len(test_samples)}')

dist = Counter([s['polarity'] for s in train_samples])
label_names = ['Negative', 'Neutral', 'Positive']
for k, v in sorted(dist.items()):
    print(f'  {label_names[k]}: {v}')

print('\n Build BERT dataset (dep parse ~3-5 phút)...')
train_dataset = BertABSADataset(train_samples, tokenizer, MAX_LEN)
test_dataset  = BertABSADataset(test_samples,  tokenizer, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                           num_workers=2, pin_memory=True)
print(' Dataset sẵn sàng!')

## 5. Định nghĩa Model

Kiến trúc gồm 3 thành phần chính:

| Module | Mô tả |
|--------|-------|
| **BERT** | Encoder lấy contextual embeddings |
| **SyncreticInfoNetwork (SIN)** | Graph neural network trên dependency tree |
| **SemanticGuidedAttention (SGSA)** | Multi-head attention dùng aspect vector làm query |

In [ ]:
#  Model 
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import BertModel

class SyncreticInfoNetwork(nn.Module):
    def __init__(self, dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.layers  = nn.ModuleList([nn.Linear(dim, dim) for _ in range(num_layers)])
        self.epsilon = nn.ParameterList([nn.Parameter(torch.zeros(1)) for _ in range(num_layers)])
        self.drop    = nn.Dropout(dropout)
        self.norm    = nn.LayerNorm(dim)

    def forward(self, x, adj):
        h = x
        for i in range(len(self.layers)):
            agg = torch.bmm(adj, h)
            h   = F.relu(self.layers[i]((1 + self.epsilon[i]) * h + agg))
            h   = self.drop(h)
        return self.norm(h)


class SemanticGuidedAttention(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.H  = num_heads
        self.Dh = dim // num_heads
        self.W_q  = nn.Linear(dim, dim)
        self.W_k  = nn.Linear(dim, dim)
        self.W_v  = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)

    def forward(self, syn, asp_vec, mask=None):
        B, L, D = syn.shape
        q = self.W_q(asp_vec).unsqueeze(1).view(B, 1, self.H, self.Dh).transpose(1, 2)
        k = self.W_k(syn).view(B, L, self.H, self.Dh).transpose(1, 2)
        v = self.W_v(syn).view(B, L, self.H, self.Dh).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.Dh)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, -1e9)
        attn = self.drop(F.softmax(scores, dim=-1))
        out  = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, 1, D)
        return self.norm(self.proj(out).squeeze(1) + asp_vec)


class BertSSINClassifier(nn.Module):
    def __init__(self, bert_model_name, hidden_dim=300, num_classes=3,
                 sin_layers=2, num_heads=4, dropout=0.3):
        super().__init__()
        self.bert     = BertModel.from_pretrained(bert_model_name)
        bert_dim      = self.bert.config.hidden_size          # 768
        self.proj     = nn.Sequential(
            nn.Linear(bert_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout)
        )
        self.sin      = SyncreticInfoNetwork(hidden_dim, sin_layers, dropout)
        self.sgsa     = SemanticGuidedAttention(hidden_dim, num_heads, dropout)
        self.asp_proj = nn.Linear(bert_dim, hidden_dim)
        self.drop     = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, token_type_ids, adj, asp_mask):
        out     = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        seq_out = out.last_hidden_state   # (B, L, 768)
        cls_out = out.pooler_output       # (B, 768)

        h       = self.proj(seq_out)      # (B, L, hidden_dim)

        # Aspect vector: trung bình các token của aspect
        asp_len = asp_mask.sum(dim=1, keepdim=True).clamp(min=1)
        asp_vec = self.asp_proj(
            (seq_out * asp_mask.unsqueeze(-1)).sum(dim=1) / asp_len
        )   # (B, hidden_dim)

        # Normalise adj theo hàng
        adj_norm = adj / adj.sum(dim=-1, keepdim=True).clamp(min=1)
        h_syn   = self.sin(h, adj_norm)                      # (B, L, hidden_dim)

        attn    = self.sgsa(h_syn, asp_vec, attention_mask)  # (B, hidden_dim)
        feat    = torch.cat([attn, self.asp_proj(cls_out)], dim=-1)
        logits  = self.classifier(self.drop(feat))           # (B, 3)
        return logits


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device: {device}')

HIDDEN_DIM  = 300
NUM_CLASSES = 3

model = BertSSINClassifier(BERT_MODEL, HIDDEN_DIM, NUM_CLASSES).to(device)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f' Total params: {n:,}')

## 6. Focal Loss

Thay thế CrossEntropyLoss để xử lý class imbalance (neutral class thường ít hơn rất nhiều).  
`gamma=2.0` theo paper gốc Lin et al. (2017)  RetinaNet.

In [16]:
#  FIX: Focal Loss 
class FocalLoss(nn.Module):
    """
    Focal Loss = -alpha_t * (1 - p_t)^gamma * log(p_t)
    Giúp focus vào các mẫu khó (neutral class) thay vì bị dominant bởi positive.
    gamma=2 là giá trị phổ biến từ paper gốc (Lin et al., 2017).
    """
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weight    = weight
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        log_prob = F.log_softmax(logits, dim=-1)
        prob     = torch.exp(log_prob)
        # Lấy prob của class đúng
        pt = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        # Focal weight
        focal_w = (1 - pt) ** self.gamma
        # Cross-entropy per sample
        ce = F.nll_loss(log_prob, targets, weight=self.weight, reduction='none')
        loss = focal_w * ce
        if self.reduction == 'mean':
            return loss.mean()
        return loss.sum()

## 7. Training  Restaurant Domain

- AdamW với learning rate khác nhau cho BERT (`2e-5`) và các layer khác (`1e-3`)
- Linear warmup scheduler (10% steps)
- Early stopping theo Macro-F1 với `patience=4`

In [ ]:
#  Training setup 
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup
import time
import numpy as np
import torch
import torch.nn as nn

NUM_EPOCHS    = 15
PATIENCE      = 4
EARLY_METRIC  = 'f1'

label_arr = np.array([s['polarity'] for s in train_samples])
weights   = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=label_arr)
print(f'  Class weights: Neg={weights[0]:.3f} | Neu={weights[1]:.3f} | Pos={weights[2]:.3f}')

criterion = FocalLoss(
    weight=torch.tensor(weights, dtype=torch.float).to(device),
    gamma=2.0
)

optimizer = torch.optim.AdamW([
    {'params': list(model.bert.parameters()), 'lr': 2e-5, 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if 'bert' not in n],
     'lr': 1e-3, 'weight_decay': 1e-4},
])

total_steps = len(train_loader) * NUM_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

#  Evaluate (FIX: thêm latency) 
def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for b in loader:
            logits = model(
                b['input_ids'].to(device),
                b['attention_mask'].to(device),
                b['token_type_ids'].to(device),
                b['adj'].to(device),
                b['asp_mask'].to(device)
            )
            preds.extend(logits.argmax(-1).cpu().numpy())
            targets.extend(b['polarity'].numpy())
    acc = accuracy_score(targets, preds)
    f1  = f1_score(targets, preds, average='macro', zero_division=0)
    return acc, f1, targets, preds

def measure_latency_batch1(model, dataset, device, n_runs=100, warmup=10):
    model.eval()
    loader_b1 = DataLoader(dataset, batch_size=1, shuffle=False)
    latencies = []
    with torch.no_grad():
        for i, b in enumerate(loader_b1):
            if i >= warmup + n_runs:
                break
            inputs = (
                b['input_ids'].to(device),
                b['attention_mask'].to(device),
                b['token_type_ids'].to(device),
                b['adj'].to(device),
                b['asp_mask'].to(device),
            )
            if i < warmup:
                _ = model(*inputs)
                continue
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(*inputs)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000)
    avg = sum(latencies) / len(latencies)
    p50 = sorted(latencies)[len(latencies) // 2]
    p95 = sorted(latencies)[int(len(latencies) * 0.95)]
    p99 = sorted(latencies)[int(len(latencies) * 0.99)]
    print(f" Latency batch=1: Mean={avg:.2f}ms | P50={p50:.2f}ms | P95={p95:.2f}ms | P99={p99:.2f}ms")
    return avg, p50, p95, p99


#  Training loop 
history = {'loss': [], 'acc': [], 'f1': []}
best_acc, best_f1, best_state = 0.0, 0.0, None
patience_counter = 0

print(f'\n{"Epoch":>6} | {"Loss":>8} | {"Acc":>7} | {"F1":>7} | {"Time":>6}')
print('-' * 52)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for b in train_loader:
        optimizer.zero_grad()
        logits = model(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['token_type_ids'].to(device),
            b['adj'].to(device),
            b['asp_mask'].to(device)
        )
        loss = criterion(logits, b['polarity'].to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    acc, f1, _, _ = evaluate(model, test_loader, device)

    history['loss'].append(avg_loss)
    history['acc'].append(acc)
    history['f1'].append(f1)

    if f1 > best_f1:
        best_f1, best_acc = f1, acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = ' '
    else:
        patience_counter += 1
        marker = ''

    elapsed = time.time() - t0
    print(f'{epoch:>6} | {avg_loss:>8.4f} | {acc:>7.4f} | {f1:>7.4f} | {elapsed:>5.1f}s{marker}')

    if patience_counter >= PATIENCE:
        print(f'\n  Early stopping tại epoch {epoch}')
        break

print(f'\n Best Accuracy: {best_acc:.4f} | Best Macro-F1: {best_f1:.4f}')


#  Final evaluation 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

model.load_state_dict(best_state)

acc, f1, targets, preds = evaluate(model, test_loader, device)

print('=' * 60)
print(f'  BERT-SSIN v2  SemEval-2014 ({DATASET.capitalize()})')
print('=' * 60)
print(classification_report(targets, preds,
      target_names=['Negative', 'Neutral', 'Positive'], digits=4))

#  Plot 
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['loss'], 'b-o', markersize=4)
axes[0].set_title('Training Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(history['acc'], 'g-o', label='Accuracy')
axes[1].plot(history['f1'], 'r-s', label='F1')
axes[1].legend()
axes[1].grid(alpha=0.3)

cm = confusion_matrix(targets, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Neg', 'Neu', 'Pos'],
            yticklabels=['Neg', 'Neu', 'Pos'])

plt.tight_layout()
plt.show()


#  Metrics JSON (FIX latency) 
import json as _json

lat_avg, lat_p50, lat_p95, lat_p99 = measure_latency_batch1(
    model, test_dataset, device, n_runs=100, warmup=10
)

metrics_rest = {
    'method': 'BERT-SSIN',
    'paradigm': 'Fine-Tuning',
    'backbone': 'bert-base-uncased',
    'dataset': 'SemEval-2014-Restaurant',
    'n_samples': len(test_samples),
    'sentiment_accuracy': round(acc, 4),
    'sentiment_macro_f1': round(f1, 4),
    'parse_error_rate': 0.0,
    'evaluation_protocol': 'given_aspect_absc',
    'given_aspect': True,
    'avg_latency_ms': round(lat_avg, 3),
    'latency_p50_ms': round(lat_p50, 3),
    'latency_p95_ms': round(lat_p95, 3),
    'latency_p99_ms': round(lat_p99, 3),   #  FIX
    'efficiency': {
        'params_million': None,
        'gpu_mem_peak_gb': None,
        'training_hours': None,
        'hardware': '1x T4 16GB',
        'precision': 'fp32',
        'batch_size_inference': 1,
    }
}

print(_json.dumps(metrics_rest, indent=2, ensure_ascii=False))

with open('/kaggle/working/bert_ssin_restaurant_metrics.json', 'w', encoding='utf-8') as f:
    _json.dump(metrics_rest, f, indent=2, ensure_ascii=False)

print(' Saved metrics with latency')

## 8. Lưu model tốt nhất

In [ ]:
#  Save model 
torch.save(
    {'model_state': best_state, 'best_acc': best_acc, 'best_f1': best_f1},
    '/kaggle/working/bert_ssin_v2_best.pt'
)
print(' Saved: bert_ssin_v2_best.pt')

## 9. Demo Inference

Test nhanh với một số câu mẫu để kiểm tra model hoạt động đúng.

In [ ]:
#  Demo inference 
def predict(text, aspect, model, tokenizer, device, max_len=128):
    model.eval()
    text_l   = text.lower()
    aspect_l = aspect.lower()
    enc = tokenizer(
        text_l, aspect_l,
        max_length=max_len, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    ids = enc['input_ids'].to(device)
    sep = (ids[0] == tokenizer.sep_token_id).nonzero(as_tuple=True)[0]
    asp_mask = torch.zeros(1, max_len).to(device)
    if len(sep) >= 2:
        asp_mask[0, sep[0] + 1:sep[1]] = 1.0

    # FIX: dùng build_dep_adj_bert
    adj = torch.tensor(
        [build_dep_adj_bert(text_l, tokenizer, max_len)],
        dtype=torch.float
    ).to(device)

    with torch.no_grad():
        logits = model(
            ids,
            enc['attention_mask'].to(device),
            enc['token_type_ids'].to(device),
            adj,
            asp_mask
        )
        prob = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        pred = logits.argmax(-1).item()
    return ['Negative ', 'Neutral ', 'Positive '][pred], prob

print('\n Demo inference:')
print('-' * 65)
for text, aspect in [
    ('The food was absolutely amazing and delicious.',      'food'),
    ('The service was slow and the waiter was very rude.',  'service'),
    ('The price is reasonable for this area.',              'price'),
    ('The laptop battery lasts only 2 hours.',              'battery'),
    ('The screen display is incredibly sharp and vivid.',   'screen'),
]:
    label, probs = predict(text, aspect, model, tokenizer, device)
    print(f'Text   : "{text}"')
    print(f'Aspect : [{aspect}]    {label}')
    print(f'Probs  : Neg={probs[0]:.3f} | Neu={probs[1]:.3f} | Pos={probs[2]:.3f}')
    print('-' * 65)

## 10. Training  Laptop Domain

Lặp lại quy trình training cho domain Laptop với cùng hyperparameters.

In [ ]:
#  Laptop dataset 
train2 = read_raw('data/laptop_train.raw')
test2  = read_raw('data/laptop_test.raw')
print(f'\n Laptop  Train: {len(train2)} | Test: {len(test2)}')

train_loader2 = DataLoader(
    BertABSADataset(train2, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
test_loader2 = DataLoader(
    BertABSADataset(test2, tokenizer, MAX_LEN),
    batch_size=BATCH_SIZE, num_workers=2
)

#  Loss + Model 
weights2 = compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2]),
    y=np.array([s['polarity'] for s in train2])
)

criterion2 = FocalLoss(
    weight=torch.tensor(weights2, dtype=torch.float).to(device),
    gamma=2.0
)

model2 = BertSSINClassifier(BERT_MODEL, HIDDEN_DIM, NUM_CLASSES).to(device)

optimizer2 = torch.optim.AdamW([
    {'params': model2.bert.parameters(), 'lr': 2e-5, 'weight_decay': 0.01},
    {'params': [p for n, p in model2.named_parameters() if 'bert' not in n],
     'lr': 1e-3, 'weight_decay': 1e-4},
])

total2 = len(train_loader2) * NUM_EPOCHS
scheduler2 = get_linear_schedule_with_warmup(
    optimizer2,
    int(total2 * 0.1),
    total2
)

#  Training 
best_f12, best_acc2, best_state2 = 0.0, 0.0, None
patience_counter2 = 0

print(f'\n{"Epoch":>6} | {"Loss":>8} | {"Acc":>7} | {"F1":>7}')
print('-' * 37)

for epoch in range(1, NUM_EPOCHS + 1):
    model2.train()
    total_loss = 0.0

    for b in train_loader2:
        optimizer2.zero_grad()

        logits = model2(
            b['input_ids'].to(device),
            b['attention_mask'].to(device),
            b['token_type_ids'].to(device),
            b['adj'].to(device),
            b['asp_mask'].to(device)
        )

        loss = criterion2(logits, b['polarity'].to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), 1.0)

        optimizer2.step()
        scheduler2.step()
        total_loss += loss.item()

    #  FIX: evaluate trả thêm latency
    acc2, f12_val, _, _ = evaluate(model2, test_loader2, device)

    if f12_val > best_f12:
        best_f12, best_acc2 = f12_val, acc2
        best_state2 = {k: v.clone() for k, v in model2.state_dict().items()}
        patience_counter2 = 0
        marker = ' '
    else:
        patience_counter2 += 1
        marker = ''

    print(f'{epoch:>6} | {total_loss/len(train_loader2):>8.4f} | {acc2:>7.4f} | {f12_val:>7.4f}{marker}')

    if patience_counter2 >= PATIENCE:
        print(f'\n  Early stopping tại epoch {epoch}')
        break

print(f'\n Laptop Best Acc: {best_acc2:.4f} | Best Macro-F1: {best_f12:.4f}')


#  Final evaluation + latency 
model2.load_state_dict(best_state2)

acc2, f12_val, targets2, preds2 = evaluate(model2, test_loader2, device)


#  Save metrics JSON 
import json

lat_avg2, lat_p50_2, lat_p95_2, lat_p99_2 = measure_latency_batch1(
    model2, test_loader2.dataset, device, n_runs=100, warmup=10
)

print(f'\n Final Laptop  Acc={acc2:.4f} | F1={f12_val:.4f} | Latency={lat_avg2:.2f} ms')

metrics_laptop = {
    'method': 'BERT-SSIN',
    'paradigm': 'Fine-Tuning',
    'backbone': 'bert-base-uncased',
    'dataset': 'SemEval-2014-Laptop',
    'n_samples': len(test2),
    'sentiment_accuracy': round(acc2, 4),
    'sentiment_macro_f1': round(f12_val, 4),
    'parse_error_rate': 0.0,
    'evaluation_protocol': 'given_aspect_absc',
    'given_aspect': True,
    'avg_latency_ms': round(lat_avg2, 3),
    'latency_p50_ms': round(lat_p50_2, 3),
    'latency_p95_ms': round(lat_p95_2, 3),
    'latency_p99_ms': round(lat_p99_2, 3),   #  FIX
    'efficiency': {
        'params_million': None,
        'gpu_mem_peak_gb': None,
        'training_hours': None,
        'hardware': '1x T4 16GB',
        'precision': 'fp32',
        'batch_size_inference': 1,
    }
}

with open('/kaggle/working/bert_ssin_laptop_metrics.json', 'w') as f:
    json.dump(metrics_laptop, f, indent=2)

print(' Saved laptop metrics with latency')

## 11. Evaluate & In kết quả  Laptop Domain

Load best model, chạy evaluate, in classification report và metrics JSON.


In [ ]:
# Evaluate Laptop — load best state & in kết quả
import json as _json

model2.load_state_dict(best_state2)

acc2_final, f12_final, targets2, preds2 = evaluate(model2, test_loader2, device)

print('=' * 60)
print('  BERT-SSIN v2 — SemEval-2014 (Laptop)')
print('=' * 60)
print(classification_report(targets2, preds2,
      target_names=['Negative', 'Neutral', 'Positive'], digits=4))

# Đo latency (chạy lại riêng cho cell này)
lat_avg2, lat_p50_2, lat_p95_2, lat_p99_2 = measure_latency_batch1(
    model2, test_loader2.dataset, device, n_runs=100, warmup=10
)
print(f'\n Avg latency: {lat_avg2:.2f} ms/sample')   # ← đổi avg_latency2 → lat_avg2

# Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm2 = confusion_matrix(targets2, preds2)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['Neg', 'Neu', 'Pos'],
            yticklabels=['Neg', 'Neu', 'Pos'])
ax.set_title(f'Confusion Matrix — Laptop | Acc={acc2_final:.4f} | F1={f12_final:.4f}')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('/kaggle/working/bert_ssin_v2_laptop_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# Metrics JSON chuẩn benchmark — Laptop
metrics_laptop = {
    'method': 'BERT-SSIN',
    'paradigm': 'Fine-Tuning',
    'backbone': 'bert-base-uncased',
    'dataset': 'SemEval-2014-Laptop',
    'n_samples': len(test2),
    'sentiment_accuracy': round(acc2_final, 4),
    'sentiment_macro_f1': round(f12_final, 4),
    'parse_error_rate': 0.0,
    'evaluation_protocol': 'given_aspect_absc',
    'given_aspect': True,
    'avg_latency_ms': round(lat_avg2, 3),
    'latency_p50_ms': round(lat_p50_2, 3),
    'latency_p95_ms': round(lat_p95_2, 3),
    'latency_p99_ms': round(lat_p99_2, 3),
    'efficiency': {
        'params_million': None,
        'gpu_mem_peak_gb': None,
        'training_hours': None,
        'hardware': '1x T4 16GB',
        'precision': 'fp32',
        'batch_size_inference': 1,
    }
}

print('\n Metrics JSON — SemEval-2014 Laptop:')
print(_json.dumps(metrics_laptop, indent=2, ensure_ascii=False))

with open('/kaggle/working/bert_ssin_laptop_metrics.json', 'w', encoding='utf-8') as f:
    _json.dump(metrics_laptop, f, indent=2, ensure_ascii=False)

print('\n Saved: bert_ssin_laptop_metrics.json')

## 12. Lưu model Laptop tốt nhất


In [ ]:
#  Save Laptop model 
torch.save(
    {'model_state': best_state2, 'best_acc': best_acc2, 'best_f1': best_f12},
    '/kaggle/working/bert_ssin_v2_laptop_best.pt'
)
print(' Saved: bert_ssin_v2_laptop_best.pt')
